# Week 6 Session 2: Joins and analytical queries at Sunrise Stores
**Forge Academy | Data Science Foundations**

*Covers Week 6 Session 2: joins and analytical queries, the SQL Analytics Challenge, and a first look at the capstone.*

## Mission

Last session we could count, filter and summarise one table at a time. Ngozi's real questions cross tables: "which categories actually brought in money?", "who are the highest-spending customers?", "are any products sitting on the shelf unsold?"

Today we join the tables, we check the grain of the result so we do not double-count, and we answer those questions in SQL. By the end you will have the brief for the SQL Analytics Challenge, and a first look at the capstone that runs through the rest of the course.

Keep `sunrise_stores.db` in the same folder as this notebook and run the cells from top to bottom.


## 0. Reconnect


In [ ]:
import sqlite3


In [ ]:
import pandas as pd


In [ ]:
conn = sqlite3.connect("sunrise_stores.db")


In [ ]:
pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name;", conn)


## 1. Why we join, and what grain means

Each table has a **key**. `customer_id` appears on `customers` and on `orders`. `order_id` appears on `orders`, `order_items` and `payments`. `product_id` appears on `products` and on `order_items`. A **join** uses a key to line the tables up.

**Grain** means: what does one row represent?

- In `orders`, one row is one basket.
- In `order_items`, one row is one line of that basket (for example, two bags of rice).
- One order often has several lines. So if you join `orders` to `order_items`, you no longer have one row per basket. You have one row per line.

That is not a mistake, unless you then treat the result as if it were still one row per basket. The habit is: **count the rows before the join, count them after, and say what a row means now.**


First, the counts we already trust.


In [ ]:
query = """
SELECT 'orders' AS table_name, COUNT(*) AS n_rows FROM orders
UNION ALL
SELECT 'customers', COUNT(*) FROM customers
UNION ALL
SELECT 'order_items', COUNT(*) FROM order_items
UNION ALL
SELECT 'products', COUNT(*) FROM products;
"""


In [ ]:
pd.read_sql(query, conn)


## 2. INNER JOIN: keep the matches

An `INNER JOIN` keeps only rows that find a match on both sides. We start with `orders` and `customers`. Each order has exactly one customer, so the result should still have one row per order: 420 rows.


In [ ]:
query = """
SELECT
    o.order_id,
    o.order_date,
    o.status,
    o.channel,
    c.first_name,
    c.last_name,
    c.city,
    c.segment
FROM orders AS o
INNER JOIN customers AS c
    ON o.customer_id = c.customer_id
LIMIT 8;
"""


In [ ]:
pd.read_sql(query, conn)


The grain check: how many rows does the join produce, compared with 420 orders?


In [ ]:
query = """
SELECT COUNT(*) AS n_rows
FROM orders AS o
INNER JOIN customers AS c
    ON o.customer_id = c.customer_id;
"""


In [ ]:
pd.read_sql(query, conn)


`AS o` and `AS c` are short names for the tables, so `o.order_id` is readable. The count should be 420. We did not inflate anything, because each order matches one customer.

Now join `orders` to `order_items`. Each basket can have several lines, so we should expect more than 420 rows.


In [ ]:
query = """
SELECT COUNT(*) AS n_rows
FROM orders AS o
INNER JOIN order_items AS oi
    ON o.order_id = oi.order_id;
"""


In [ ]:
pd.read_sql(query, conn)


That result should be 1,071, the same as the number of order items. The grain is now **one row per line**, not one row per order. If we counted rows here and called it "number of orders", we would be wrong.

`SELECT *` after a join is especially noisy, because every column from both tables appears, and duplicated keys are easy to miss. Name the columns you need.


### Your turn 1

1. Join `order_items` to `products` on `product_id`. Show `item_id`, `product_name`, `category`, `quantity` and `order_items.unit_price` for 8 rows.
2. Count the rows in that join. Does the number match 1,071? What is the grain of the result?


In [ ]:
# Your code here


In [ ]:
# Your code here


## 3. LEFT JOIN: keep the left table, even when there is no match

An `INNER JOIN` hides absences. If a customer never placed an order, they disappear from an inner join of customers to orders. A `LEFT JOIN` keeps every row from the **left** table, and puts empty values (`NULL`) on the right when there is no match.

How many customers have no orders? We start from `customers` and left-join `orders`.


In [ ]:
query = """
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    c.city,
    c.segment,
    o.order_id
FROM customers AS c
LEFT JOIN orders AS o
    ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL
ORDER BY c.customer_id;
"""


In [ ]:
no_orders = pd.read_sql(query, conn)


In [ ]:
no_orders


In [ ]:
print(len(no_orders))


`IS NULL` means "this value is empty". Those customers signed up and have not (yet) placed an order. There should be 12 of them, and they are all in the New segment, which makes sense.

The same pattern finds products that were never ordered.


In [ ]:
query = """
SELECT
    p.product_id,
    p.product_name,
    p.category,
    p.unit_price
FROM products AS p
LEFT JOIN order_items AS oi
    ON p.product_id = oi.product_id
WHERE oi.product_id IS NULL
ORDER BY p.product_id;
"""


In [ ]:
pd.read_sql(query, conn)


### Your turn 2

Are there any **Completed** orders that have no payment? Start from `orders`, keep only `status = 'Completed'`, left-join `payments`, and keep rows where `payments.payment_id` is empty. Show the count of such orders. (A result of zero is a real answer, if you can say what it means.)


In [ ]:
# Your code here


## 4. Analytical queries: revenue, with a careful grain

Ngozi's question: **which product categories brought in the most money from completed orders?**

We need three tables: line items (quantity and price), products (category), and orders (status). Revenue for a line is `quantity * unit_price`. We keep only `Completed` orders, then group by category.


In [ ]:
query = """
SELECT
    p.category,
    SUM(oi.quantity) AS units,
    SUM(oi.quantity * oi.unit_price) AS revenue
FROM order_items AS oi
INNER JOIN products AS p
    ON p.product_id = oi.product_id
INNER JOIN orders AS o
    ON o.order_id = oi.order_id
WHERE o.status = 'Completed'
GROUP BY p.category
ORDER BY revenue DESC;
"""


In [ ]:
revenue_by_category = pd.read_sql(query, conn)


In [ ]:
revenue_by_category


Staples should be far ahead, which fits a provisions shop. Cancelled and refunded baskets are excluded, because they are not kept sales.

A second question: **who are the ten customers who have spent the most on completed orders?** This time the grain of the *answer* is one row per customer, even though we pass through orders and items to get there.


In [ ]:
query = """
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    c.city,
    c.segment,
    COUNT(DISTINCT o.order_id) AS n_orders,
    SUM(oi.quantity * oi.unit_price) AS spend
FROM customers AS c
INNER JOIN orders AS o
    ON o.customer_id = c.customer_id
INNER JOIN order_items AS oi
    ON oi.order_id = o.order_id
WHERE o.status = 'Completed'
GROUP BY c.customer_id, c.first_name, c.last_name, c.city, c.segment
ORDER BY spend DESC
LIMIT 10;
"""


In [ ]:
pd.read_sql(query, conn)


`COUNT(DISTINCT o.order_id)` is needed because the join to `order_items` would otherwise count lines, not baskets. That is grain awareness in one function.


### Your turn 3

Using only completed orders, compare **Staples** and **Snacks**: total units sold and total revenue for each. You can filter with `WHERE p.category IN ('Staples', 'Snacks')`.


In [ ]:
# Your code here


## 5. Average order value, the careful way

"What is the average order value by channel?" sounds simple, and it is the classic place to get the grain wrong.

If you join orders to items and then take `AVG(quantity * unit_price)` grouped by channel, you are averaging **line values**, not basket totals. A four-line order would be treated as four small orders.

The safe pattern is two steps in one query:

1. Build a small inner table with **one row per completed order** and its total value.
2. Average those totals by channel.


In [ ]:
query = """
SELECT
    channel,
    COUNT(*) AS n_orders,
    ROUND(AVG(order_value), 1) AS avg_order_value,
    SUM(order_value) AS revenue
FROM (
    SELECT
        o.order_id,
        o.channel,
        SUM(oi.quantity * oi.unit_price) AS order_value
    FROM orders AS o
    INNER JOIN order_items AS oi
        ON oi.order_id = o.order_id
    WHERE o.status = 'Completed'
    GROUP BY o.order_id, o.channel
)
GROUP BY channel
ORDER BY avg_order_value DESC;
"""


In [ ]:
pd.read_sql(query, conn)


The part in parentheses is a **subquery**: a query that builds a temporary table, here with one row per completed order. Then the outer query averages those rows.

Here is the wrong-grain version, on purpose, so you can see how different the numbers look. This averages line values, not order values.


In [ ]:
query = """
SELECT
    o.channel,
    COUNT(*) AS n_lines,
    ROUND(AVG(oi.quantity * oi.unit_price), 1) AS avg_line_value
FROM orders AS o
INNER JOIN order_items AS oi
    ON oi.order_id = o.order_id
WHERE o.status = 'Completed'
GROUP BY o.channel;
"""


In [ ]:
pd.read_sql(query, conn)


The wrong query has a much smaller "average" and a `COUNT` that is a line count, not an order count. When a number looks like average order value but is only a few thousand naira in a shop that sells 10 kg rice, check the grain.


### Your turn 4

For completed orders, show each payment `method` with the number of payments and the total `amount`. Order by total amount, largest first. You will join `payments` to `orders` so you can keep only completed orders.


In [ ]:
# Your code here


## 6. Challenge (10 to 15 minutes)

1. For completed orders, show **monthly** revenue. Use `strftime('%Y-%m', o.order_date) AS month` to build a year-month label, then group by that label. Show the month, the number of distinct orders, and the revenue, in calendar order. Which month is highest?
2. List customers with **more than 8** completed orders. Show `customer_id`, `first_name`, `last_name` and the order count, highest first.
3. In one or two sentences (a markdown cell is fine): why is the "average line value by channel" query not a safe measure of average order value?


In [ ]:
# Your code here


In [ ]:
# Your code here


**Your answer to question 3:** (write yours here)


## Recap

- A join lines tables up on a shared key. `INNER JOIN` keeps matches only. `LEFT JOIN` keeps every row from the left table and uses `IS NULL` to find absences.
- Always check the grain: count rows before and after, and say what one row means now.
- Money questions should filter to `status = 'Completed'` unless you have a reason not to.
- `COUNT(DISTINCT order_id)` and "sum first, then average" are two practical ways to stay at the grain you meant.
- A subquery can build a one-row-per-order table before you summarise it.

## The SQL Analytics Challenge

This is the major Week 6 assignment (10% of the course). The full brief is in `Week6_SQL_Analytics_Challenge.txt` in this folder. You will receive a starter notebook and a `queries.sql` template as well.

In short: answer the numbered business questions with SQL, in a file called `queries.sql` with short comments, plus a README that says how to run the queries, plus a results write-up that explains what each answer means. Grain-aware joins and clear `GROUP BY` matter more than clever SQL.

## A first look at the capstone

From this week you can start thinking about the end-of-course capstone (25% of the course). The brief is in the Week 8 folder as `Week8_Capstone_Brief.txt`; a short copy of the idea is also in this week's assignment message.

The capstone is one end-to-end project, using tools you already have (Python, pandas, EDA, charts, statistics, SQL, and later a simple model if it fits). You choose a problem and a dataset. You will not need a new method in Week 8; that week is for finishing, writing, and presenting.

A complete capstone is shaped like this:

Problem, Dataset, Prep, EDA, Stats, Model (if you use one), Evaluation, Insights, Recommendations, Limitations, Future work.

Deliverables in Week 8: a GitHub repository, a notebook that runs from top to bottom, a README, a 7 to 10 minute presentation, and a short written reflection.

This week, a small step is enough: write down one possible question and one possible dataset. You can change your mind later.


## Assignment

**Due:** before our Week 7 Session 1 class. **Submit:** `queries.sql`, a README, and your results / explanation, through the usual submission form.

Follow `Week6_SQL_Analytics_Challenge.txt`. Use `sunrise_stores.db` (SQLite is the expected path for this class). PostgreSQL is not required.

Also, in your README or in a short extra note, write one sentence for a possible capstone question and one sentence for a dataset you might use. This is a starting thought, not a locked decision.

**Rubric (20 points, 4 points each, 10% of the course):** Technical correctness, Problem solving, Analytical reasoning, Documentation, Communication. Full details are in the brief. Join logic and grain awareness come first.

AI tools are allowed, but you must be able to explain every query and every number on our call.
